# Thesis: Reclaimed Timber in Deep Generative Design
**Notebook ID:** 23_timber_stock_dataset_generation  
**Author:** Jasper Cluistra   
**Last Updated:** 2026-02-24
### Properties script
**Goal:** to generate the timber stock datasets, that the algorithm can use to choose elements for the design

**Inputs:**
*   parameters for properties of the stock
*   search space of geometry

**Outputs:**
*   CSV file with dataset of new stock
*   CSV file with dataset of reused stock

# PARAMETERS

In [6]:
# ==========================================
# CEL 1: ALLE INPUT PARAMETERS (NEW & RECLAIMED)
# ==========================================
import itertools
import random
import numpy as np
import pandas as pd

# --- PARAMETERS NEW STOCK (CATALOGUS) - EFFICIËNTE DEFINITIE ---
TUPLE_LENGTHS = (2400, 2700, 3000, 3300, 3600, 4000, 4400, 4800, 5200)

# DEPTH_WIDTH_MAPPING: Voor elke depth, welke widths zijn geldig
DEPTH_WIDTH_MAPPING = {
    100: [38, 50, 63, 75, 100],
    150: [38, 50, 63, 75, 100],
    175: [38, 50, 63, 75],
    200: [38, 50, 63, 75, 100, 150, 200],
    225: [38, 50, 63, 75],
    250: [50, 75, 100, 250],
    300: [50, 75, 100, 150, 300]
}

# Auto-genereer DEPTH_WIDTH_COMBINATIONS uit DEPTH_WIDTH_MAPPING
DEPTH_WIDTH_COMBINATIONS = [
    (depth, width)
    for depth in DEPTH_WIDTH_MAPPING.keys()
    for width in DEPTH_WIDTH_MAPPING[depth]
]

MECH_PROPS_NEW = {
    'E_modulus_eff': 11000.0, # N/mm2
    'f_mk': 24,               # N/mm2
    'Density': 420            # kg/m3
}

# LCA Aannames Nieuw Hout
LCA_NEW = {
    'Embodied Carbon Coëfficiënt': 150.0,  # Fictief hoog (productie + drogen)
    'Transport_Dist': 1500,                 # Vaste grotere afstand (bijv. import)
    'Emmisiefactor_diesel_range': (0.08, 0.15), # Alleen groot diesel transport voor nieuw hout
    'Bewerkingsfactor': 0                  # 0 = Geen ontspijkering nodig
}

# --- PARAMETERS RECLAIMED STOCK ---
DONOR_BATCHES = [
    {"batch_id": "B01", "count": 2, "orig_width": 180, "orig_depth": 600, "orig_length": 12000},
    {"batch_id": "B02", "count": 10, "orig_width": 75, "orig_depth": 225, "orig_length": 5400},
    {"batch_id": "B03", "count": 4, "orig_width": 200, "orig_depth": 200, "orig_length": 4200}
]

MECH_PROPS_RECLAIMED = {
    'C24': {'e_mod': 11000.0, 'f_mk': 24, 'density': 420},
    'C18': {'e_mod': 9000.0,  'f_mk': 18, 'density': 380}
}

# LCA Aannames Reclaimed Hout
LCA_RECLAIMED = {
    'Embodied Carbon Coëfficiënt': 15.0,   # Fictief laag (enkel de-constructie impact)
    'Emmisiefactor_diesel_range': (0.08, 0.15),
    'Emmisiefactor_elektrisch_range': (0.02, 0.05),
    'Kans_op_elektrisch': 0.30, # 30% kans dat het lokaal via een e-truck gaat
    'Bewerkingsfactor': 1                  # 1 = Ontspijkeren en schaven nodig
}
# LCA: Willekeurige afstand vanaf donor site
# Locatie Delft: (lokaal, Groningen)
transport_dist = random.randint(10, 240)

print("Alle parameters succesvol geladen! Ga naar cel 2.")

Alle parameters succesvol geladen! Ga naar cel 2.


# GENERATION

## New Stock

In [7]:
# ==========================================
# CEL 2: GENERATIE NEW STOCK CATALOGUS
# ==========================================
import itertools
import random
import pandas as pd

def generate_new_timber_catalog():
    data = []

    combinaties = list(itertools.product(TUPLE_LENGTHS, DEPTH_WIDTH_COMBINATIONS))
    print(f"📊 Catalogus genereren... {len(combinaties)} balk-typen")

    for index, (length, (depth, width)) in enumerate(combinaties):
        emmisiefactor_new = random.uniform(*LCA_NEW['Emmisiefactor_diesel_range'])

        data.append({
            'Member_ID': f"NS_{index:05d}",
            'State': 0,
            'Length_Actual': float(length),
            'Depth': float(depth),
            'Width': float(width),
            'E_modulus_eff': float(MECH_PROPS_NEW['E_modulus_eff']),
            'f_mk': int(MECH_PROPS_NEW['f_mk']),
            'Density': int(MECH_PROPS_NEW['Density']),
            'Embodied Carbon Coëfficiënt': float(LCA_NEW['Embodied Carbon Coëfficiënt']),
            'Transport_Dist': int(LCA_NEW['Transport_Dist']),
            'Emmisiefactor': round(emmisiefactor_new, 4),
            'Bewerkingsfactor': int(LCA_NEW['Bewerkingsfactor'])
        })

    df_new = pd.DataFrame(data)
    print("✅ New stock succesvol gegenereerd!")
    return df_new

df_new_stock = generate_new_timber_catalog()
display(df_new_stock.head(3))

# ==========================================
# CEL 3: TABEL MET DEPTH EN WIDTH
# ==========================================
print("\n" + "="*50)
print("📊 TABEL: ALLE DIEPTE EN BREEDTE COMBINATIES")
print("="*50)

depth_width_combinations = df_new_stock[['Depth', 'Width']].drop_duplicates().sort_values(by=['Depth', 'Width']).reset_index(drop=True)

print(f"\n{'Basic size':<15} {'D (mm)':<12} {'B (mm)':<12}")
print("-" * 40)

for idx, (_, row) in enumerate(depth_width_combinations.iterrows(), 1):
    depth = int(row['Depth'])
    width = int(row['Width'])
    print(f"{idx:<15} {depth:<12} {width:<12}")

print("-" * 40)
print(f"Totaal aantal combinaties: {len(depth_width_combinations)}")

depth_width_combinations.to_csv('depth_width_combinations.csv', index=False)
print("\n✅ Geëxporteerd naar 'depth_width_combinations.csv'")

📊 Catalogus genereren... 315 balk-typen
✅ New stock succesvol gegenereerd!


,Member_ID,State,Length_Actual,Depth,Width,E_modulus_eff,f_mk,Density,Embodied Carbon Coëfficiënt,Transport_Dist,Emmisiefactor,Bewerkingsfactor
0,NS_00000,0,2400.0,100.0,38.0,11000.0,24,420,150.0,1500,0.1026,0
1,NS_00001,0,2400.0,100.0,50.0,11000.0,24,420,150.0,1500,0.1099,0
2,NS_00002,0,2400.0,100.0,63.0,11000.0,24,420,150.0,1500,0.0824,0



📊 TABEL: ALLE DIEPTE EN BREEDTE COMBINATIES

Basic size      D (mm)       B (mm)      
----------------------------------------
1               100          38          
2               100          50          
3               100          63          
4               100          75          
5               100          100         
6               150          38          
7               150          50          
8               150          63          
9               150          75          
10              150          100         
11              150          150         
12              175          38          
13              175          50          
14              175          63          
15              175          75          
16              200          38          
17              200          50          
18              200          63          
19              200          75          
20              200          100         
21              200          15

## Reused Stock

In [ ]:
# ==========================================
# CEL 3: GENERATIE RECLAIMED STOCK
# ==========================================
def generate_reclaimed_stock():
    inventory_list = []
    current_id_number = 1

    for batch in DONOR_BATCHES:
        for _ in range(batch['count']):
            # Geometrie: Sloop- en schaafverlies
            cut_loss = random.randint(100, 400)
            length_actual = batch['orig_length'] - cut_loss
            depth = batch['orig_depth'] - random.randint(10, 16)
            width = batch['orig_width'] - random.randint(10, 16)

            # Grading (Kwaliteit) bepalen
            grade = np.random.choice(['C24', 'C18'], p=[0.60, 0.40])

            # LCA: Transport afstand en dynamische emissiefactor
            transport_dist = random.randint(20, 150)

            # Bepaal of dit specifieke element met een elektrische of diesel truck gaat
            if random.random() < LCA_RECLAIMED['Kans_op_elektrisch']:
                emmisiefactor_reclaimed = random.uniform(*LCA_RECLAIMED['Emmisiefactor_elektrisch_range'])
            else:
                emmisiefactor_reclaimed = random.uniform(*LCA_RECLAIMED['Emmisiefactor_diesel_range'])

            inventory_list.append({
                "Member_ID": f"RS_{current_id_number:05d}",
                "State": 1, # 1 = Reclaimed

                # Geometrie
                "Length_Actual": float(length_actual),
                "Depth": float(depth),
                "Width": float(width),

                # Mechanisch
                "E_modulus_eff": float(MECH_PROPS_RECLAIMED[grade]['e_mod']),
                "f_mk": int(MECH_PROPS_RECLAIMED[grade]['f_mk']),
                "Density": int(MECH_PROPS_RECLAIMED[grade]['density']),

                # LCA
                "Embodied Carbon Coëfficiënt": float(LCA_RECLAIMED['Embodied Carbon Coëfficiënt']),
                "Transport_Dist": int(transport_dist),
                "Emmisiefactor": round(emmisiefactor_reclaimed, 4), # Afgerond op 4 decimalen
                "Bewerkingsfactor": int(LCA_RECLAIMED['Bewerkingsfactor'])
            })
            current_id_number += 1

    df_reused = pd.DataFrame(inventory_list)
    print(f"Reclaimed stock gegenereerd! Totaal elementen: {len(df_reused)}")
    return df_reused

df_reused_stock = generate_reclaimed_stock()
display(df_reused_stock.head(3))

Reclaimed stock gegenereerd! Totaal elementen: 16


,Member_ID,State,Length_Actual,Depth,Width,E_modulus_eff,f_mk,Density,Embodied Carbon Coëfficiënt,Transport_Dist,Emmisiefactor,Bewerkingsfactor
0,RS_00001,1,11746.0,588.0,168.0,9000.0,18,380,15.0,93,0.0498,1
1,RS_00002,1,11709.0,585.0,168.0,11000.0,24,420,15.0,34,0.0350,1
2,RS_00003,1,5288.0,214.0,60.0,11000.0,24,420,15.0,138,0.0280,1


# EXPORTING

In [ ]:
from config import RAW_DATA_PATH
# ==========================================
# CEL 4: SAMENVOEGEN EN EXPORTEREN
# ==========================================
# Voeg beide datasets samen tot één grote matrix
df_combined_stock = pd.concat([df_new_stock, df_reused_stock], ignore_index=True)

# Toon een willekeurige selectie uit de gecombineerde dataset
print("\nPreview van de gecombineerde inventaris (mix van NS en RS):")
display(df_combined_stock.sample(6))

# Exporteren
dataset_filename = 'complete_timber.csv'
df_combined_stock.to_csv(RAW_DATA_PATH + dataset_filename, index=False, sep=';')
# df_combined_stock.to_csv(dataset_filename, index=False, sep=';')

print(f"\nAlles succesvol samengevoegd en opgeslagen als '{dataset_filename}'.")


Preview van de gecombineerde inventaris (mix van NS en RS):


,Member_ID,State,Length_Actual,Depth,Width,E_modulus_eff,f_mk,Density,Embodied Carbon Coëfficiënt,Transport_Dist,Emmisiefactor,Bewerkingsfactor
51,NS_00051,0,2700.0,250.0,100.0,11000.0,24,420,150.0,500,0.1492,0
162,NS_00162,0,4000.0,250.0,75.0,11000.0,24,420,150.0,500,0.1416,0
156,NS_00156,0,4000.0,225.0,38.0,11000.0,24,420,150.0,500,0.1263,0
155,NS_00155,0,4000.0,200.0,100.0,11000.0,24,420,150.0,500,0.1442,0
58,NS_00058,0,3000.0,100.0,75.0,11000.0,24,420,150.0,500,0.0817,0
171,RS_00004,1,5277.0,215.0,59.0,9000.0,18,380,15.0,95,0.1003,1



Alles succesvol samengevoegd en opgeslagen als 'complete_timber.csv'.
